# Data Transformation

This notebook reads the E-SMAT Bronze Delta tables, applies data typing, normalization, enrichment, and business transformations, and writes standardized Silver datasets.

## Responsibilities

1. Load reusable functions;
2. Configure Bronze sources and Silver targets;
3. Read and validate Bronze tables;
4. Transform inspection requirements;
5. Transform inspection headers;
6. Transform question-level responses;
7. Enrich responses with requirement metadata;
8. Create stable analytical keys;
9. Persist Silver Delta tables.

## Outputs

- `inspection_header_silver`
- `inspection_responses_silver`
- `inspection_requirements_silver`

> Detailed data-quality controls are executed in `04_data_quality.ipynb`. This notebook applies only the structural checks required to transform and persist the Silver datasets.


## 1. Load shared functions

In [ ]:
%run ./01_shared_functions


## 2. Configure Bronze sources and Silver targets

Replace the placeholders only in the private production version.


In [ ]:
BRONZE_CATALOG = "<catalog>"
BRONZE_SCHEMA = "<bronze-schema>"
SILVER_CATALOG = "<catalog>"
SILVER_SCHEMA = "<silver-schema>"

TABLE_CONFIG = {
    "inspection_header": {
        "source_table": (
            f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.inspection_header_bronze"
        ),
        "target_table": (
            f"{SILVER_CATALOG}.{SILVER_SCHEMA}.inspection_header_silver"
        ),
    },
    "inspection_responses": {
        "source_table": (
            f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.inspection_responses_bronze"
        ),
        "target_table": (
            f"{SILVER_CATALOG}.{SILVER_SCHEMA}.inspection_responses_silver"
        ),
    },
    "inspection_requirements": {
        "source_table": (
            f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.inspection_requirements_bronze"
        ),
        "target_table": (
            f"{SILVER_CATALOG}.{SILVER_SCHEMA}.inspection_requirements_silver"
        ),
    },
}

WRITE_MODE = "overwrite"
OVERWRITE_SCHEMA = "true"


## 3. Validate table configuration

In [ ]:
def validate_table_config(table_config):
    if not table_config:
        raise ValueError("TABLE_CONFIG cannot be empty.")

    required_datasets = {
        "inspection_header",
        "inspection_responses",
        "inspection_requirements",
    }
    missing_datasets = sorted(required_datasets - set(table_config))

    if missing_datasets:
        raise ValueError(
            f"Missing transformation datasets: {missing_datasets}"
        )

    source_tables = []
    target_tables = []

    for dataset_name, dataset_config in table_config.items():
        for config_key in ("source_table", "target_table"):
            table_name = dataset_config.get(config_key)
            if not table_name or table_name.count(".") != 2:
                raise ValueError(
                    f"Dataset '{dataset_name}' has an invalid {config_key}: "
                    f"{table_name}"
                )

        source_tables.append(dataset_config["source_table"])
        target_tables.append(dataset_config["target_table"])

    if len(source_tables) != len(set(source_tables)):
        raise ValueError("Bronze source tables must be unique.")
    if len(target_tables) != len(set(target_tables)):
        raise ValueError("Silver target tables must be unique.")

    print(
        f"Transformation configuration validated for "
        f"{len(table_config)} datasets."
    )


validate_table_config(TABLE_CONFIG)


## 4. Read Bronze tables

The Bronze layer preserves source-aligned values as strings. This notebook converts the fields required by the analytical model to explicit Spark data types.


In [ ]:
bronze_dataframes = {}

for dataset_name, dataset_config in TABLE_CONFIG.items():
    source_table = dataset_config["source_table"]

    if not spark.catalog.tableExists(source_table):
        raise RuntimeError(
            f"Bronze source table not found for '{dataset_name}': "
            f"{source_table}"
        )

    dataframe = spark.table(source_table)

    if len(dataframe.columns) != len(set(dataframe.columns)):
        raise ValueError(
            f"Bronze dataset '{dataset_name}' contains duplicate columns."
        )

    bronze_dataframes[dataset_name] = dataframe
    print(f"Bronze dataset loaded: {dataset_name}")


## 5. Shared transformation helpers

These functions are specific to the E-SMAT business model, so they remain in the transformation notebook rather than the generic shared-functions notebook.


In [ ]:
def normalize_boolean(column_name):
    normalized_value = F.lower(F.trim(F.col(column_name).cast("string")))

    return (
        F.when(
            normalized_value.isin("true", "sim", "sí", "1", "yes"),
            F.lit(True),
        )
        .when(
            normalized_value.isin("false", "não", "nao", "no", "0"),
            F.lit(False),
        )
        .otherwise(F.lit(None).cast("boolean"))
    )


def normalize_theme(column_name):
    normalized_value = F.lower(F.trim(F.col(column_name)))

    return (
        F.when(
            normalized_value.isin(
                "env",
                "environment",
                "environmental",
                "meio ambiente",
            ),
            F.lit("environment"),
        )
        .when(
            normalized_value.isin(
                "saf",
                "safety",
                "segurança",
                "seguranca",
            ),
            F.lit("safety"),
        )
        .otherwise(normalized_value)
    )


def normalize_shift(column_name):
    normalized_value = F.lower(F.trim(F.col(column_name)))

    return (
        F.when(
            normalized_value.isin("adm", "administrativo"),
            F.lit("ADM"),
        )
        .when(
            normalized_value.isin("primeiro", "primer turno", "1"),
            F.lit("1"),
        )
        .when(
            normalized_value.isin("segundo", "segundo turno", "2"),
            F.lit("2"),
        )
        .when(
            normalized_value.isin("terceiro", "tercer turno", "3"),
            F.lit("3"),
        )
        .otherwise(F.trim(F.col(column_name)))
    )


def normalize_activity_type(column_name):
    normalized_value = F.lower(F.trim(F.col(column_name)))

    return (
        F.when(
            normalized_value.isin(
                "rotineira",
                "rutinaria",
                "rutinária",
            ),
            F.lit("Rotineira"),
        )
        .when(
            normalized_value.isin(
                "não-rotineira",
                "nao-rotineira",
                "não rotineira",
                "nao rotineira",
                "no rutinaria",
            ),
            F.lit("Não-rotineira"),
        )
        .otherwise(F.trim(F.col(column_name)))
    )


def normalize_contract_type(column_name):
    normalized_value = F.lower(F.trim(F.col(column_name)))

    return (
        F.when(
            normalized_value.isin(
                "empregado interno",
                "empleado interno",
                "internal employee",
            ),
            F.lit("Empregado interno"),
        )
        .when(
            normalized_value.isin(
                "terceiro",
                "contratista",
                "contractor",
            ),
            F.lit("Terceiro"),
        )
        .otherwise(F.trim(F.col(column_name)))
    )


def parse_source_date(column_name):
    return F.coalesce(
        F.to_date(F.col(column_name), "dd/MM/yyyy"),
        F.to_date(F.col(column_name), "yyyy-MM-dd"),
        F.to_date(F.substring(F.col(column_name), 1, 10), "yyyy-MM-dd"),
    )


def keep_numeric_identifier(column_name):
    trimmed_value = F.trim(F.col(column_name).cast("string"))
    return F.when(
        trimmed_value.rlike(r"^[0-9]+$"),
        trimmed_value,
    ).otherwise(F.lit(None).cast("string"))


## 6. Transform inspection requirements

The requirements dataset becomes the reference dimension used to enrich question-level responses with methodology, theme, category, version, and active-status metadata.


In [ ]:
requirements_bronze_df = bronze_dataframes["inspection_requirements"]

require_columns(
    dataframe=requirements_bronze_df,
    required_columns={
        "site",
        "methodology",
        "questionnaire_version",
        "theme",
        "category_id",
        "category_name_pt_br",
        "category_name_es_ar",
        "question_id",
        "question_column_id",
        "question_number",
        "requirement_text",
        "active",
    },
    dataset_name="inspection_requirements_bronze",
)

requirements_silver_df = (
    requirements_bronze_df
    .withColumn("site", F.trim(F.col("site")))
    .withColumn("methodology", F.upper(F.trim(F.col("methodology"))))
    .withColumn("questionnaire_version", F.trim(F.col("questionnaire_version")))
    .withColumn("theme", normalize_theme("theme"))
    .withColumn("category_id", F.trim(F.col("category_id")))
    .withColumn("category_name_pt_br", F.trim(F.col("category_name_pt_br")))
    .withColumn("category_name_es_ar", F.trim(F.col("category_name_es_ar")))
    .withColumn("question_id", F.trim(F.col("question_id")))
    .withColumn("question_column_id", F.trim(F.col("question_column_id")))
    .withColumn("question_number", F.col("question_number").cast("int"))
    .withColumn("requirement_text", F.trim(F.col("requirement_text")))
    .withColumn("active", normalize_boolean("active"))
    .withColumn(
        "requirement_key",
        F.concat_ws(
            "_",
            F.coalesce(F.col("site"), F.lit("")),
            F.coalesce(F.col("methodology"), F.lit("")),
            F.coalesce(F.col("questionnaire_version"), F.lit("")),
            F.coalesce(F.col("question_id"), F.lit("")),
        ),
    )
    .dropDuplicates(["requirement_key"])
)

requirements_silver_df = add_processing_timestamp(
    requirements_silver_df.drop("processing_timestamp")
)

print("Inspection requirements transformed successfully.")


## 7. Transform inspection headers

The header dataset is standardized at one row per inspection. Dates and Boolean fields are typed, bilingual categorical values are normalized, and identifiers required by the analytical model are derived.


In [ ]:
header_bronze_df = bronze_dataframes["inspection_header"]

require_columns(
    dataframe=header_bronze_df,
    required_columns={
        "inspection_id",
        "site",
        "inspection_datetime",
        "shift",
        "segment",
        "business_unit_area",
        "line",
        "workstation",
        "activity_type",
        "contract_type",
        "observed_person_id_1",
        "job_role",
        "observer_id",
        "observed_person_id_2",
        "observed_person_id_3",
        "unsafe_condition",
        "unsafe_condition_description",
        "safety_tag_opened",
        "safety_tag_number",
        "near_miss",
        "near_miss_description",
        "audit_type",
        "created_by",
    },
    dataset_name="inspection_header_bronze",
)

header_silver_df = (
    header_bronze_df
    .withColumn("inspection_id", F.trim(F.col("inspection_id")))
    .withColumn("site", F.trim(F.col("site")))
    .withColumn(
        "inspection_date",
        parse_source_date("inspection_datetime"),
    )
    .withColumn("inspection_year", F.year(F.col("inspection_date")))
    .withColumn("shift", normalize_shift("shift"))
    .withColumn("segment", F.trim(F.col("segment")))
    .withColumn("business_unit_area", F.trim(F.col("business_unit_area")))
    .withColumn("line", F.trim(F.col("line")))
    .withColumn("workstation", F.trim(F.col("workstation")))
    .withColumn("activity_type", normalize_activity_type("activity_type"))
    .withColumn("contract_type", normalize_contract_type("contract_type"))
    .withColumn("observer_id", keep_numeric_identifier("observer_id"))
    .withColumn(
        "observed_person_id_1",
        keep_numeric_identifier("observed_person_id_1"),
    )
    .withColumn(
        "observed_person_id_2",
        keep_numeric_identifier("observed_person_id_2"),
    )
    .withColumn(
        "observed_person_id_3",
        keep_numeric_identifier("observed_person_id_3"),
    )
    .withColumn("unsafe_condition", normalize_boolean("unsafe_condition"))
    .withColumn("safety_tag_opened", normalize_boolean("safety_tag_opened"))
    .withColumn("near_miss", normalize_boolean("near_miss"))
    .withColumn(
        "audit_type",
        F.coalesce(
            F.when(
                F.trim(F.col("audit_type")) != "",
                F.upper(F.trim(F.col("audit_type"))),
            ),
            F.regexp_extract(
                F.col("inspection_id"),
                r"^(EMAT|SMAT|ESMAT)",
                1,
            ),
        ),
    )
)

observed_id_columns = [
    "observed_person_id_1",
    "observed_person_id_2",
    "observed_person_id_3",
]

observed_people_count = F.lit(0)
for column_name in observed_id_columns:
    observed_people_count = observed_people_count + F.when(
        F.col(column_name).isNotNull(),
        F.lit(1),
    ).otherwise(F.lit(0))

header_silver_df = (
    header_silver_df
    .withColumn("observed_people_count", observed_people_count)
    .withColumn(
        "observed_people_ids",
        F.concat_ws(
            ", ",
            *[
                F.when(
                    F.col(column_name).isNotNull(),
                    F.col(column_name),
                )
                for column_name in observed_id_columns
            ],
        ),
    )
    .withColumn(
        "header_key",
        F.concat_ws(
            "_",
            F.coalesce(F.col("inspection_id"), F.lit("")),
            F.coalesce(F.col("inspection_date").cast("string"), F.lit("")),
        ),
    )
    .dropDuplicates(["inspection_id"])
)

header_silver_df = add_processing_timestamp(
    header_silver_df.drop("processing_timestamp")
)

print("Inspection headers transformed successfully.")


## 8. Transform and enrich inspection responses

Question-level responses are typed and enriched with the active questionnaire metadata. The header date is used as the authoritative inspection date.


In [ ]:
responses_bronze_df = bronze_dataframes["inspection_responses"]

require_columns(
    dataframe=responses_bronze_df,
    required_columns={
        "inspection_id",
        "site",
        "question_id",
        "safe_act",
        "unsafe_act",
        "description",
        "created_by",
    },
    dataset_name="inspection_responses_bronze",
)

responses_typed_df = (
    responses_bronze_df
    .withColumn("inspection_id", F.trim(F.col("inspection_id")))
    .withColumn("site", F.trim(F.col("site")))
    .withColumn("question_id", F.trim(F.col("question_id")))
    .withColumn("safe_act", F.coalesce(F.col("safe_act").cast("int"), F.lit(0)))
    .withColumn("unsafe_act", F.coalesce(F.col("unsafe_act").cast("int"), F.lit(0)))
    .withColumn("description", F.trim(F.col("description")))
)

header_reference_df = header_silver_df.select(
    "inspection_id",
    "inspection_date",
    "audit_type",
).dropDuplicates(["inspection_id"])

requirement_reference_df = (
    requirements_silver_df
    .filter(F.col("active") == F.lit(True))
    .select(
        "site",
        "methodology",
        "questionnaire_version",
        "theme",
        "category_id",
        "category_name_pt_br",
        "category_name_es_ar",
        "question_id",
        "question_number",
        "requirement_text",
    )
)

response_alias = responses_typed_df.alias("response")
header_alias = header_reference_df.alias("header")
requirement_alias = requirement_reference_df.alias("requirement")

responses_with_header_df = response_alias.join(
    header_alias,
    F.col("response.inspection_id") == F.col("header.inspection_id"),
    how="left",
).select(
    *[
        F.col(f"response.{column_name}").alias(column_name)
        for column_name in responses_typed_df.columns
        if column_name not in {"inspection_date", "theme"}
    ],
    F.col("header.inspection_date").alias("inspection_date"),
    F.col("header.audit_type").alias("audit_type"),
)

responses_silver_df = (
    responses_with_header_df.alias("response")
    .join(
        requirement_alias,
        (
            F.col("response.site") == F.col("requirement.site")
        )
        & (
            F.col("response.question_id") == F.col("requirement.question_id")
        )
        & (
            F.col("response.audit_type") == F.col("requirement.methodology")
        ),
        how="left",
    )
    .select(
        *[
            F.col(f"response.{column_name}").alias(column_name)
            for column_name in responses_with_header_df.columns
        ],
        F.col("requirement.questionnaire_version"),
        F.col("requirement.theme"),
        F.col("requirement.category_id"),
        F.col("requirement.category_name_pt_br"),
        F.col("requirement.category_name_es_ar"),
        F.col("requirement.question_number"),
        F.col("requirement.requirement_text"),
    )
    .withColumn(
        "response_key",
        F.concat_ws(
            "_",
            F.coalesce(F.col("inspection_id"), F.lit("")),
            F.coalesce(F.col("question_id"), F.lit("")),
        ),
    )
    .dropDuplicates(["response_key"])
)

responses_silver_df = add_processing_timestamp(
    responses_silver_df.drop("processing_timestamp")
)

print("Inspection responses transformed and enriched successfully.")


## 9. Review transformed schemas and row counts

These are structural checks only. Referential integrity, accepted-value checks, conditional-field rules, and duplicate diagnostics are handled in `04_data_quality.ipynb`.


In [ ]:
silver_dataframes = {
    "inspection_header": {
        "df": header_silver_df,
        "target_table": TABLE_CONFIG["inspection_header"]["target_table"],
    },
    "inspection_responses": {
        "df": responses_silver_df,
        "target_table": TABLE_CONFIG["inspection_responses"]["target_table"],
    },
    "inspection_requirements": {
        "df": requirements_silver_df,
        "target_table": TABLE_CONFIG["inspection_requirements"]["target_table"],
    },
}

required_silver_columns = {
    "inspection_header": {
        "inspection_id",
        "inspection_date",
        "inspection_year",
        "audit_type",
        "header_key",
        "processing_timestamp",
    },
    "inspection_responses": {
        "inspection_id",
        "question_id",
        "inspection_date",
        "safe_act",
        "unsafe_act",
        "theme",
        "response_key",
        "processing_timestamp",
    },
    "inspection_requirements": {
        "requirement_key",
        "question_id",
        "theme",
        "active",
        "processing_timestamp",
    },
}

transformation_summary = []

for dataset_name, dataset_info in silver_dataframes.items():
    dataframe = dataset_info["df"]

    require_columns(
        dataframe=dataframe,
        required_columns=required_silver_columns[dataset_name],
        dataset_name=f"{dataset_name}_silver",
    )

    if len(dataframe.columns) != len(set(dataframe.columns)):
        raise ValueError(
            f"Silver dataset '{dataset_name}' contains duplicate columns."
        )

    record_count = dataframe.count()

    transformation_summary.append(
        {
            "dataset": dataset_name,
            "records": record_count,
            "columns": len(dataframe.columns),
            "target_table": dataset_info["target_table"],
            "status": "transformed",
        }
    )

transformation_summary_df = spark.createDataFrame(transformation_summary)
display(transformation_summary_df.orderBy("dataset"))


## 10. Persist Silver Delta tables

In [ ]:
load_summary = []

for dataset_name, dataset_info in silver_dataframes.items():
    dataframe = dataset_info["df"]
    target_table = dataset_info["target_table"]
    record_count = next(
        item["records"]
        for item in transformation_summary
        if item["dataset"] == dataset_name
    )

    print(f"Writing Silver dataset '{dataset_name}' to {target_table}...")

    (
        dataframe.write.format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", OVERWRITE_SCHEMA)
        .saveAsTable(target_table)
    )

    load_summary.append(
        {
            "dataset": dataset_name,
            "target_table": target_table,
            "records": record_count,
            "status": "written",
        }
    )

    print(f"Silver table written successfully: {record_count} records.")

silver_load_summary_df = spark.createDataFrame(load_summary)
display(silver_load_summary_df.orderBy("dataset"))

print("E-SMAT Silver transformation completed successfully.")


## Next step

Run `04_data_quality.ipynb` to evaluate uniqueness, referential integrity, accepted values, conditional fields, null keys, date conversion, and questionnaire enrichment across the Silver datasets.
